In [1]:
import os

In [2]:
from  dotenv import load_dotenv

In [3]:
load_dotenv()

True

In [4]:
keyvalue = os.getenv("KEYNAME")
print(f"keyvalue => {keyvalue}")

keyvalue => None


In [5]:
from langchain_groq import ChatGroq

In [6]:
llm_model = ChatGroq(model="qwen/qwen3-32b")

In [7]:
llm_model.invoke("what is latest trend in AI field today")

AIMessage(content="<think>\nOkay, so I need to figure out the latest trends in AI as of now. Let me start by recalling what I know about AI developments. There's been a lot of talk about large language models like GPT, BERT, and their variants. Companies like Google, Meta, and Anthropic have been pushing the boundaries with models that are getting bigger and more capable, right? I remember reading that they're using techniques like transformer architectures and scaling up parameters to improve performance.\n\nThen there's the area of multimodal AI. I think that's when AI combines different types of data like text, images, and audio. For example, you can have a model that not only understands text but also processes images or videos. Maybe tools like DALL-E or MidJourney are part of this trend. They generate images from text descriptions, which is pretty cool. I wonder how that's evolving now.\n\nAnother thing I heard about is AI in healthcare. There's been progress in diagnostics, drug

In [8]:
from langchain_community.document_loaders import PyPDFLoader,TextLoader, Docx2txtLoader

In [9]:
from pathlib import Path

In [10]:
file_directory = Path("./data/multi_chat")

In [11]:
all_document=[]
for file_name in file_directory.iterdir():
    if not file_name.is_file():
        continue
    file_extension = (Path(file_name.name).suffix.lower())
    if file_extension == ".docx":
        loader = Docx2txtLoader(file_name)
    elif file_extension == ".pdf":
        loader = PyPDFLoader(file_name)
    elif file_extension == ".txt":
        loader = TextLoader(file_name)
    else:
        continue

    all_document.extend(loader.load())



        
    

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [13]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=300)
chunks = splitter.split_documents(all_document)

In [14]:
from langchain_community.vectorstores import FAISS
# from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.embeddings import OpenAIEmbeddings

In [15]:
# embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
# embeddings = OllamaEmbeddings(model="mxbai-embed-large")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
	


C:\Users\Admin\AppData\Local\Temp\ipykernel_11948\3735232309.py:3: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embeddings = OpenAIEmbeddings(model="text-embedding-3-large")


In [16]:
FAISS_INDEX_PATH = Path("faiss_index")

In [17]:
if FAISS_INDEX_PATH.exists():
    vector_store = FAISS.load_local(folder_path=str(FAISS_INDEX_PATH),embeddings=embeddings, allow_dangerous_deserialization=True)
    print("retrieved faiss index")
else:
    vector_store = FAISS.from_documents(documents=chunks, embedding=embeddings)
    # vectors = [embeddings.embed_query(doc.page_content) for doc in chunks]
    # vector_store = FAISS.from_embeddings(embeddings=vectors, documents=chunks)
    vector_store.save_local(str("faiss_index"))
    print("created faiss index")

retrieved faiss index


In [18]:
# retriever = vector.as_retriever()
retriever =vector_store.as_retriever()

In [19]:
from langchain_core.prompts import  ChatPromptTemplate, MessagesPlaceholder

In [51]:
qa_prompt = ChatPromptTemplate.from_messages([
("system",(
    "You are an assistant designed to provide answer using the provided context "
    "Relay your answer only on the retrieved information.  If you don't have answer then "
    "reply that you don't have information. Keep you answer more concise and no longer three sentences. \n\n {context}"
)),
MessagesPlaceholder("chat_history_test"),
("human","{input}")
])


In [52]:
# Prompt for contextual question rewriting
contextualize_the_question_prompt = ChatPromptTemplate.from_messages([
("system",(
    "Given a conversational history and recent user query, rewrite the user question as standalone "
    "question which doesn't require previous context.  Don't answer the user question and rewrite the question, if required, "
    " otherwise return the query without change"
)),

MessagesPlaceholder("chat_history_test"),
("human","{input}")

])

In [39]:
from langchain.chains import create_history_aware_retriever, create_retrieval_chain

In [53]:
history_aware_retriver = create_history_aware_retriever(llm_model,retriever , contextualize_the_question_prompt) 

In [54]:
from langchain.chains.combine_documents import create_stuff_documents_chain

In [55]:
qa_chain = create_stuff_documents_chain(llm_model,qa_prompt)

In [56]:
rag_chain = create_retrieval_chain(
    history_aware_retriver,
    qa_chain
)

In [27]:
from langchain_core.runnables import RunnableWithMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

import streamlit as st

In [57]:
def _get_message_history(session_id:str) -> BaseChatMessageHistory:
    if "store" not in st.session_state:
        st.session_state.store={}
    if session_id not in st.session_state.store:
        st.session_state.store[session_id]=ChatMessageHistory()
    return st.session_state.store[session_id]


In [58]:
session_id = "test_conversational_rag"

1) 🧩 What history_messages_key="chat_history_test" Means
    This tells LangChain:
    “When building the prompt, insert the messages from _get_message_history() into the placeholder called chat_history_test.”

2) {input}, {context}, chat_history — these are not reserved keywords. You define them, and LangChain matches them based on:
    Prompt template
    Chain configuration
    Message history manager

In [59]:
chain = RunnableWithMessageHistory(
    rag_chain,
    _get_message_history,
    input_messages_key="input",
    history_messages_key="chat_history_test",
    output_messages_key="answer"
)

In [60]:
# user_input="what is the main topic of the document?"
user_input ="what is President Zelenskyy said in their speech in parliament?"

In [46]:
from typing import Optional, List
from langchain_core.messages import BaseMessage

In [61]:
# chat_history:Optional[List[BaseMessage]]=None

response = chain.invoke(
{"input":user_input},
config={"configurable":{"session_id":session_id}}
)

2025-08-13 20:22:38.915 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-13 20:22:38.917 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-13 20:22:38.918 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [34]:
answer = response.get("answer","No answer.")

In [35]:
answer

'<think>\nOkay, let me try to figure out the answer. The user is asking what President Zelenskyy said in his speech to the European Parliament. The provided context mentions that in his speech, he said, “Light will win over darkness.” \n\nFirst, I need to check if that\'s the exact quote. The context says: "as President Zelenskyy said in his speech to the European Parliament \'Light will win over darkness.\'" So that\'s the specific quote given in the context.\n\nI need to make sure there\'s no other information about his speech in the provided context. The rest of the text talks about the speaker addressing Congress, mentioning Zelenskyy\'s courage and the Ukrainian people\'s resistance, but the only direct quote from Zelenskyy is about light overcoming darkness. \n\nSince the user is asking specifically about his speech in parliament, but the context refers to the European Parliament. The user might be conflating the European Parliament with Ukraine\'s parliament, but the context cle